# FBSA — Evaluation & Visualization

Loads `best_model.pt`, computes test Dice/IoU/HD on BRISC test split, shows 10 random `image | gt | pred` triplets.

In [ ]:
import sys, random
from pathlib import Path
PROJECT = Path('..').resolve()
sys.path.insert(0, str(PROJECT))
import torch, kagglehub, matplotlib.pyplot as plt
from tumor_seg.config import TrainConfig
from tumor_seg.data import create_brisc_dataloaders
from tumor_seg.metrics import dice_coefficient, iou_coefficient, hausdorff_distance_metric
from tumor_seg.models import FBSASegmenter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = '/content/drive/MyDrive/fbsa_runs/v1/best_model.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = TrainConfig(**ckpt['cfg'])
model = FBSASegmenter(
    encoder_name=cfg.encoder, encoder_dim=cfg.encoder_dim,
    num_slots=cfg.num_slots, slot_dim=cfg.slot_dim,
    slot_iters=cfg.slot_iters, slot_hidden=cfg.slot_hidden,
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print('best epoch =', ckpt['epoch'])

In [ ]:
data_root = str(Path(kagglehub.dataset_download('briscdataset/brisc2025')) / 'brisc2025' / 'segmentation_task')
_, val_loader = create_brisc_dataloaders(data_root, batch_size=cfg.batch_size, image_size=cfg.image_size)

tot_d=tot_i=tot_h=0.0; n=0
all_imgs, all_gts, all_preds = [], [], []
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        bs = images.size(0); n += bs
        tot_d += dice_coefficient(logits, masks).item() * bs
        tot_i += iou_coefficient(logits, masks).item() * bs
        tot_h += hausdorff_distance_metric(logits, masks, image_size=cfg.image_size) * bs
        all_imgs.append(images.cpu()); all_gts.append(masks.cpu())
        all_preds.append((torch.sigmoid(logits) > 0.5).float().cpu())
print(f'Test Dice = {tot_d/n:.4f}  IoU = {tot_i/n:.4f}  HD = {tot_h/n:.2f} px')

In [ ]:
imgs = torch.cat(all_imgs); gts = torch.cat(all_gts); preds = torch.cat(all_preds)
k = min(10, imgs.shape[0])
idxs = random.sample(range(imgs.shape[0]), k)
fig, axes = plt.subplots(k, 3, figsize=(9, 3*k))
for r, i in enumerate(idxs):
    img = imgs[i].permute(1,2,0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    axes[r,0].imshow(img); axes[r,0].set_title('image'); axes[r,0].axis('off')
    axes[r,1].imshow(gts[i,0], cmap='gray'); axes[r,1].set_title('gt'); axes[r,1].axis('off')
    axes[r,2].imshow(preds[i,0], cmap='gray'); axes[r,2].set_title('pred'); axes[r,2].axis('off')
plt.tight_layout(); plt.show()